# 10 — Daten zurücksetzen und Infrastruktur herunterfahren

## Zweck
Den Projektzustand sauber zurücksetzen: alle von den Notebooks erzeugten Dateien unter `data/` löschen
und — lokal — die Docker-Infrastruktur herunterfahren. Danach kann die Pipeline (Notebooks `00`–`09`)
aus einem sauberen Zustand neu starten.

Geschützt bleiben `.gitkeep`-Dateien (Verzeichnisstruktur) und der Ordner `data/test-executed-notebooks/`.
Alle gelöschten Daten lassen sich durch erneutes Ausführen der Pipeline wiederherstellen.

## Konfiguration

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
import subprocess

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=False)
DATA_DIR = PROJECT_ROOT / "data"
EXECUTION_ENV = os.getenv("EXECUTION_ENV", "docker_compose")

PROTECTED_NAMES = {".gitkeep"}
PROTECTED_DIR = "test-executed-notebooks"
print({"execution_env": EXECUTION_ENV, "data_dir": str(DATA_DIR)})

## Generierte Dateien löschen

In [ ]:
deleted = 0
for path in DATA_DIR.rglob("*"):
    if not path.is_file() or path.name in PROTECTED_NAMES:
        continue
    if PROTECTED_DIR in {p.name for p in path.parents}:
        continue
    path.unlink()
    deleted += 1
print(f"{deleted} Dateien geloescht.")

## Infrastruktur herunterfahren
Lokal stoppt `docker compose down` Kafka, Spark, PostgreSQL und Jupyter (die PostgreSQL-Daten im Volume
bleiben erhalten; für einen vollständigen Reset `docker compose down -v` im Terminal verwenden).
Auf der FH gibt es keine eigene Infrastruktur zum Stoppen.

In [ ]:
if EXECUTION_ENV == "docker_compose":
    subprocess.run(["docker", "compose", "down"], cwd=PROJECT_ROOT, check=True)
    print("Infrastruktur heruntergefahren (docker compose down).")
else:
    print("FH-Umgebung: keine lokale Infrastruktur zum Herunterfahren.")

## Fertig
Der Projektzustand ist zurückgesetzt. Für einen neuen Lauf mit Notebook `00` beginnen.